# GRID Gemma 4 — Train All 4 Tasks (Single Load)

Trains all 4 micro-models in one session without reloading.
Saves LoRA adapters to HuggingFace. GGUF conversion happens on the GRID server.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl==0.22.2

In [ ]:
import os, sys, gc, torch

HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not os.path.exists('/content/grid'):
    !git clone https://github.com/3pacs/GRID.git /content/grid
    %cd /content/grid
else:
    %cd /content/grid
    !git pull

sys.path.insert(0, '/content/grid')

# Load HF token from .env if available
if not HF_TOKEN:
    try:
        with open('/content/grid/.env') as f:
            for line in f:
                if line.startswith('HF_API_KEY='):
                    HF_TOKEN = line.strip().split('=', 1)[1]
                    os.environ['HF_TOKEN'] = HF_TOKEN
                    break
    except FileNotFoundError:
        pass

print(f'HF Token: {"configured" if HF_TOKEN else "NOT SET"}')

In [ ]:
# === CONFIG ===
BASE_MODEL = "unsloth/gemma-4-E2B-it"
TASKS = ["signal_classifier", "anomaly_narrator", "edgar_extractor", "knowledge_mapper"]
HF_USERNAME = "stepdadfinance"
MAX_SEQ_LENGTH = 2048
LORA_R = 8
LORA_ALPHA = 8
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10

In [ ]:
# Train all 4 tasks sequentially, save LoRA adapters
import os, sys
os.chdir('/content/grid')
if '/content/grid' not in sys.path:
    sys.path.insert(0, '/content/grid')

from unsloth.chat_templates import standardize_data_formats, train_on_responses_only
from gemma.training.config import TaskType, TASK_SYSTEM_PROMPTS
from gemma.training.datasets import load_dataset_for_training
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer
import torch

test_prompts = {
    "signal_classifier": "Breaking: Federal Reserve announced emergency 50bp rate cut. Treasury yields dropping sharply.",
    "anomaly_narrator": "Feature: SPX_DAILY_RETURN\nValue: -6.8%\nExpected: -0.1%\nZ-score: -5.2\nPeriod: 2026-04-05",
    "edgar_extractor": "Extract: company_name, filing_type, total_revenue\n\nAMAZON.COM INC\nFORM 10-Q\nNet revenue: $155.7 billion",
    "knowledge_mapper": "BlackRock increased Bitcoin ETF holdings to $45B while lobbying SEC for spot Ethereum ETF approval.",
}

results = {}

for task_name in TASKS:
    print(f"\n{'='*60}")
    print(f"  TASK: {task_name}")
    print(f"{'='*60}\n")

    # Attach fresh LoRA
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        random_state=3407,
    )

    # Load dataset
    task = TaskType(task_name)
    dataset = load_dataset_for_training(task)
    dataset = standardize_data_formats(dataset)

    def format_conversations(examples):
        convos = examples["conversations"]
        texts = [
            tokenizer.apply_chat_template(
                convo, tokenize=False, add_generation_prompt=False
            ).removeprefix("<bos>")
            for convo in convos
        ]
        return {"text": texts}

    dataset = dataset.map(format_conversations, batched=True)
    print(f"Dataset: {len(dataset)} examples")

    # Train
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=None,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_steps=WARMUP_STEPS,
            num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=f"outputs/{task_name}",
            report_to="none",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<start_of_turn>user\n",
        response_part="<start_of_turn>model\n",
    )

    stats = trainer.train()
    loss = stats.training_loss
    print(f"\nTraining done — loss: {loss:.4f}")

    # Quick inference test
    print(f"\n--- Inference test ---")
    sys_prompt = TASK_SYSTEM_PROMPTS[task]
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": test_prompts[task_name]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    _ = model.generate(
        **inputs, max_new_tokens=256,
        temperature=0.1, top_p=0.95,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    # Save LoRA adapter
    lora_dir = f"outputs/{task_name}/lora"
    model.save_pretrained(lora_dir)
    tokenizer.save_pretrained(lora_dir)
    print(f"\nLoRA saved to {lora_dir}")

    # Push to HuggingFace
    if HF_TOKEN:
        repo = f"{HF_USERNAME}/grid-gemma4-{task_name}-lora"
        print(f"Pushing to {repo}...")
        model.push_to_hub(repo, token=HF_TOKEN)
        tokenizer.push_to_hub(repo, token=HF_TOKEN)
        print(f"Pushed: https://huggingface.co/{repo}")
    else:
        print("HF_TOKEN not set — skipping push")

    results[task_name] = {"loss": loss, "examples": len(dataset)}

    # Cleanup for next task
    del trainer, dataset, stats
    gc.collect()
    torch.cuda.empty_cache()

    # Unload LoRA for next task
    try:
        model = model.merge_and_unload()
    except Exception:
        pass  # Some versions handle this differently

print(f"\n\n{'='*60}")
print("ALL TASKS COMPLETE")
print(f"{'='*60}")
for t, r in results.items():
    print(f"  {t}: loss={r['loss']:.4f}, examples={r['examples']}")

In [ ]:
# Train all 4 tasks sequentially, save LoRA adapters
from unsloth.chat_templates import standardize_data_formats, train_on_responses_only
from gemma.training.config import TaskType, TASK_SYSTEM_PROMPTS
from gemma.training.datasets import load_dataset_for_training
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer
import torch

test_prompts = {
    "signal_classifier": "Breaking: Federal Reserve announced emergency 50bp rate cut. Treasury yields dropping sharply.",
    "anomaly_narrator": "Feature: SPX_DAILY_RETURN\nValue: -6.8%\nExpected: -0.1%\nZ-score: -5.2\nPeriod: 2026-04-05",
    "edgar_extractor": "Extract: company_name, filing_type, total_revenue\n\nAMAZON.COM INC\nFORM 10-Q\nNet revenue: $155.7 billion",
    "knowledge_mapper": "BlackRock increased Bitcoin ETF holdings to $45B while lobbying SEC for spot Ethereum ETF approval.",
}

results = {}

for task_name in TASKS:
    print(f"\n{'='*60}")
    print(f"  TASK: {task_name}")
    print(f"{'='*60}\n")

    # Attach fresh LoRA
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        random_state=3407,
    )

    # Load dataset
    task = TaskType(task_name)
    dataset = load_dataset_for_training(task)
    dataset = standardize_data_formats(dataset)

    def format_conversations(examples):
        convos = examples["conversations"]
        texts = [
            tokenizer.apply_chat_template(
                convo, tokenize=False, add_generation_prompt=False
            ).removeprefix("<bos>")
            for convo in convos
        ]
        return {"text": texts}

    dataset = dataset.map(format_conversations, batched=True)
    print(f"Dataset: {len(dataset)} examples")

    # Train
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=None,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_steps=WARMUP_STEPS,
            num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=f"outputs/{task_name}",
            report_to="none",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<start_of_turn>user\n",
        response_part="<start_of_turn>model\n",
    )

    stats = trainer.train()
    loss = stats.training_loss
    print(f"\nTraining done — loss: {loss:.4f}")

    # Quick inference test
    print(f"\n--- Inference test ---")
    sys_prompt = TASK_SYSTEM_PROMPTS[task]
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": test_prompts[task_name]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    _ = model.generate(
        **inputs, max_new_tokens=256,
        temperature=0.1, top_p=0.95,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    # Save LoRA adapter
    lora_dir = f"outputs/{task_name}/lora"
    model.save_pretrained(lora_dir)
    tokenizer.save_pretrained(lora_dir)
    print(f"\nLoRA saved to {lora_dir}")

    # Push to HuggingFace
    if HF_TOKEN:
        repo = f"{HF_USERNAME}/grid-gemma4-{task_name}-lora"
        print(f"Pushing to {repo}...")
        model.push_to_hub(repo, token=HF_TOKEN)
        tokenizer.push_to_hub(repo, token=HF_TOKEN)
        print(f"Pushed: https://huggingface.co/{repo}")
    else:
        print("HF_TOKEN not set — skipping push")

    results[task_name] = {"loss": loss, "examples": len(dataset)}

    # Cleanup for next task
    del trainer, dataset, stats
    gc.collect()
    torch.cuda.empty_cache()

    # Unload LoRA for next task
    try:
        model = model.merge_and_unload()
    except Exception:
        pass  # Some versions handle this differently

print(f"\n\n{'='*60}")
print("ALL TASKS COMPLETE")
print(f"{'='*60}")
for t, r in results.items():
    print(f"  {t}: loss={r['loss']:.4f}, examples={r['examples']}")